# Phase 3 — Modern catalog feasibility and versioned data

Retrospective next-retained-ML-event regression. Cutoff selection uses pre-2023 training data only. Validation is 2023; the newly frozen holdout is 2024 up to the original training boundary. Those events were previously used to train the old model, so this is an internal retrospective experiment, not independent external validation. The original validation/test populations are excluded. Purge boundary samples until every input and rolling-feature dependency belongs to its own partition.

Completeness method: MAXC plus GFT, adapted from Mignan & Woessner (2012), sections 6.1–6.2 and appendix (https://www.corssa.org/export/sites/corssa/.galleries/articles-pdf/Mignan-Woessner-2012-CORSSA-Magnitude-of-completeness.pdf_2063069299.pdf). MAXC is the modal magnitude bin. GFT compares observed cumulative counts with a fitted exponential tail. We require >=100 tail events, examine widths 0.1/0.2/0.3 ML, and use chronological moving-block bootstrap intervals (200 resamples, block20). These are conditional catalog-based estimates; mixed geography, magnitude reporting and dependence prevent claiming calibrated detection completeness. No magnitude-based event filtering or declustering is introduced.


In [1]:
from pathlib import Path
import json, hashlib, math, platform
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
ROOT=Path.cwd()
if ROOT.name!="bilstm new updated": ROOT=ROOT/"bilstm new updated"
ROOT=ROOT.resolve(); OLD=ROOT/"dataset/v2_timestamp_fixed"
OUT=ROOT/"phase3_bilstm_results"; NEW=OUT/"modern_dataset_v1"
OUT.mkdir(exist_ok=True); NEW.mkdir(exist_ok=True)
def sha(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
def clean(v):
    if isinstance(v,dict): return {str(k):clean(x) for k,x in v.items()}
    if isinstance(v,(tuple,list,np.ndarray)): return [clean(x) for x in v]
    if isinstance(v,np.generic): return clean(v.item())
    if isinstance(v,float) and not np.isfinite(v): return None
    return v
def savej(p,v): p.write_text(json.dumps(clean(v),indent=2,allow_nan=False),encoding="utf-8")
def readj(p): return json.loads(p.read_text(encoding="utf-8"))
config_old=readj(OLD/"preprocessing_config.json")
schema=readj(OLD/"feature_schema.json")
original_manifest=pd.read_csv(OLD/"split_manifest.csv")
original_train_end=pd.to_datetime(original_manifest.loc[original_manifest.split=="train","target_timestamp_utc"],utc=True).max()
SOURCE=(OLD/config_old["source_file"]).resolve()
assert sha(SOURCE)==config_old["source_sha256"]
protocol={
 "version":"phase3_modern_v1","source_sha256":sha(SOURCE),"seed":42,
 "train_before":"2023-01-01T00:00:00+00:00","validation_before":"2024-01-01T00:00:00+00:00",
 "holdout_through":original_train_end.isoformat(),"candidate_years_newest_first":[2020,2015,2010],
 "cutoff_selection":"First newest candidate passing fixed training-only feasibility gates; never compare validation/holdout for cutoff",
 "gates":{"min_train":500,"min_train_high":30,"min_unique_magnitudes":10,
          "max_half_mean_difference":0.35,"min_half_std_ratio":0.5,"max_half_std_ratio":2.0,
          "min_half_samples":200,"min_validation_samples":100,"min_holdout_samples":100},
 "gate_qualification":"Pragmatic exploratory sample/stability thresholds, not formal power or stationarity guarantees.",
 "purge":"All input, target and rolling dependency source events confined to partition",
 "sequence_length":20,"target":"Magnitude_Value ML, identity; no Mc censoring",
 "mc":{"widths":[0.1,0.2,0.3],"bootstrap":200,"block":20,"min_events":100,"min_tail":100,
       "method":"MAXC modal bin and GFT95 then GFT90; no automatic MAXC substitution for failed GFT",
       "grid":"centers at multiples of width; lower-closed upper-open bins, ties to upper bin"},
 "model":{"hidden_size":128,"layers":2,"dropout":0.3144753602375439,
          "learning_rate":0.0004549047669323766,"weight_decay":0.00012155233771053111,
          "batch_size":128,"optimizer":"AdamW","delta":1.0,"max_epochs":50,"patience":8,"gradient_clip":1.0},
 "sampler":"weights=1+0.5*clip((train_y-4)/1.5,0,1); replacement; n_train draws; sorted parents; numpy SeedSequence([42,epoch])",
 "selection":"Minimum validation MAE; ties favor clean; freeze before one holdout evaluation",
 "historical_holdout_exposure":"New partition within old training catalog; not previously unseen external data. Do not evaluate old model on it or compare old/new scores as proof of improvement.",
 "analysis_end":"2022-12-31; later target values never inform cutoff or preprocessing decisions"}
protocol_hash=hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()
if (OUT/"protocol.json").exists(): assert readj(OUT/"protocol.json")==protocol
else: savej(OUT/"protocol.json",protocol)
cached=(OUT/"analysis_complete.json").exists()
if cached: assert readj(OUT/"analysis_complete.json")["protocol_hash"]==protocol_hash
# Protect source datasets, original model/audit files and input notebooks; only hashes, never old model inference.
protected=readj(ROOT/"phase2_bilstm_results/gemini_verification/correction_provenance_v1.json")["protected_hashes_after"]
assert all(sha(ROOT/rel)==digest for rel,digest in protected.items())
print("Protocol locked; cache:",cached,"; original training ends",original_train_end.date())


Protocol locked; cache: True ; original training ends 2024-12-20


In [2]:
if not cached:
    raw=pd.read_csv(SOURCE,dtype={"source_id":str})
    cat=pd.read_csv(OLD/"catalog_events.csv",dtype={"source_id":str})
    assert raw.source_id.is_unique and cat.source_id.is_unique
    indexed=raw.set_index("source_id").loc[cat.source_id]
    assert (indexed.Magnitude_Type.str.strip().str.upper()=="ML").all()
    for old_col,csv_col in [("magnitude","Magnitude_Value"),("depth_km","Depth_km"),
                            ("latitude_deg","Lat"),("longitude_deg","Long")]:
        assert np.allclose(cat[old_col],pd.to_numeric(indexed[csv_col]).to_numpy(),rtol=0,atol=1e-9)
    times=pd.to_datetime(indexed.origin_time_utc,utc=True,format="mixed").reset_index(drop=True)
    assert (times==pd.to_datetime(cat.time,utc=True)).all()
    # Rebuild from original source columns using verified eligibility IDs; no v2 tensors/scalers reused.
    cat["time"]=times
    cat["original_timestamp"]=indexed.origin_time_utc.to_numpy()
    cat=cat[cat.time<=original_train_end].reset_index(drop=True)
    assert cat.time.is_monotonic_increasing and not cat.time.duplicated().any()
    mag=cat.magnitude.to_numpy(dtype=float); depth=cat.depth_km.to_numpy(dtype=float)
    ns=cat.time.dt.as_unit("ns").astype("int64").to_numpy()
    days=np.r_[np.nan,np.diff(ns)/1e9/86400]
    left=np.searchsorted(ns,ns-int(30*86400*1e9),side="right")
    rate=(np.arange(len(cat))-left+1)/30
    roll_m=pd.Series(mag).rolling(20,min_periods=1)
    roll_d=pd.Series(depth).rolling(20,min_periods=1)
    F=np.column_stack([mag,depth,cat.latitude_deg,cat.longitude_deg,days,rate,
                       roll_m.mean(),roll_m.std(ddof=0),roll_d.mean(),roll_d.std(ddof=0)])
    # Earliest dependency of any of the 20 input rows, including their rolling history and time gap.
    t=np.arange(39,len(cat)); dep=np.minimum(t-39,left[t-20])
    assert np.all(dep>=0)
    def eligible(start,end,inclusive=False):
        start=pd.Timestamp(start); end=pd.Timestamp(end)
        first=int(np.searchsorted(ns,start.value,side="left"))
        target_times=cat.time.iloc[t]
        mask=(target_times>=start)&((target_times<=end) if inclusive else (target_times<end))
        return t[np.asarray(mask)&(dep>=first)]
    # Known subsecond and open-left 30-day checks, independent of stored tensors.
    assert (pd.Timestamp("2020-01-01T00:00:00.5Z")-pd.Timestamp("2020-01-01T00:00:00Z")).total_seconds()==0.5
    toy=pd.to_datetime(["2020-01-01","2020-01-31","2020-01-31T00:00:00.5"],utc=True,format="mixed").as_unit("ns").asi8
    assert np.searchsorted(toy,toy[1]-int(30*86400*1e9),side="right")==1
    del raw,indexed
    # Snapshot only training-period analysis; no validation/holdout label summaries here.
    def mc_estimate(m,width):
        m=np.asarray(m,float)
        if len(m)<100: return {"n":len(m),"maxc":None,"gft":None,"reason":"<100 events"}
        integer=np.floor(m/width+0.5+1e-9).astype(int)
        centers=np.arange(integer.min(),integer.max()+1)*width
        counts=np.bincount(integer-integer.min(),minlength=len(centers))
        cumulative=counts[::-1].cumsum()[::-1]
        maxc=float(centers[np.argmax(counts)])
        fits=[]
        for j,mc in enumerate(centers):
            tail=m[integer>=int(round(mc/width))]
            if len(tail)<100: continue
            denom=tail.mean()-(mc-width/2)
            if denom<=0: continue
            b=np.log10(np.e)/denom
            model=len(tail)*10**(-b*(centers[j:]-mc))
            R=100*(1-np.abs(cumulative[j:]-model).sum()/cumulative[j:].sum())
            fits.append({"mc":float(mc),"fit_percent":float(R),"tail_n":len(tail),"b":float(b)})
        found=None;level=None
        for threshold in [95,90]:
            passing=[r for r in fits if r["fit_percent"]>=threshold]
            if passing: found=passing[0];level=threshold;break
        return {"n":len(m),"maxc":maxc,"gft":None if found is None else found["mc"],
                "fit_threshold_percent":level,"selected_fit":found,"curve":fits,
                "reason":"Conditional FMD estimate, not independently verified detection completeness"}
    def block_indices(n,rng,block=20):
        starts=rng.integers(0,max(1,n-block+1),size=math.ceil(n/block))
        return (starts[:,None]+np.arange(min(block,n))).reshape(-1)[:n]
    mc_rows=[]
    ranges=[("before2000",None,"2000-01-01"),("2000_2009","2000-01-01","2010-01-01"),
            ("2010_2014","2010-01-01","2015-01-01"),("2015_2019","2015-01-01","2020-01-01"),
            ("2020_2022","2020-01-01","2023-01-01")]
    for name,start,end in ranges:
        mask=cat.time<pd.Timestamp(end,tz="UTC")
        if start: mask &= cat.time>=pd.Timestamp(start,tz="UTC")
        period_mag=mag[mask]
        for width in protocol["mc"]["widths"]:
            estimate=mc_estimate(period_mag,width)
            estimate.update(period=name,width=width)
            if len(period_mag)>=100:
                rng=np.random.default_rng(42)
                bootstrap=[mc_estimate(period_mag[block_indices(len(period_mag),rng)],width) for _ in range(200)]
                for method in ["maxc","gft"]:
                    vals=[r[method] for r in bootstrap if r[method] is not None]
                    estimate[method+"_bootstrap_success_fraction"]=len(vals)/200
                    estimate[method+"_bootstrap_percentile95"]=np.quantile(vals,[.025,.975]).tolist() if vals else None
            mc_rows.append(estimate)
    savej(OUT/"completeness_analysis.json",{"method":protocol["mc"],"estimates":mc_rows,
        "uncertainty":"200 chronological moving-block resamples, block20; conditional percentile intervals, not coverage guaranteed under nonstationarity. Failed GFT replicates excluded and success fractions reported.",
        "limitations":"Regional/time heterogeneity, retained ML filtering, digitization and reporting unknowns; no formal external Mc validation."})
    # Prespecified newest acceptable target period, using training targets only.
    candidates=[];relations=[]
    for year in protocol["candidate_years_newest_first"]:
        ids=eligible(f"{year}-01-01T00:00:00+00:00",protocol["train_before"])
        yy=mag[ids];h=len(ids)//2
        ratio=float(yy[h:].std()/yy[:h].std()) if h>1 and yy[:h].std()>0 else 0
        gap=float(abs(yy[:h].mean()-yy[h:].mean())) if h else float("inf")
        gates={"n>=500":len(ids)>=500,"high>=30":int((yy>=4.5).sum())>=30,
               "unique>=10":len(np.unique(yy))>=10,"halves>=200":h>=200,
               "half_mean_difference<=0.35":gap<=.35,"half_sd_ratio_in_0.5_2":.5<=ratio<=2}
        candidates.append(dict(year=year,n=len(ids),high=int((yy>=4.5).sum()),unique=int(len(np.unique(yy))),
                               mean=float(yy.mean()) if len(yy) else None,half_mean_gap=gap,
                               half_sd_ratio=ratio,gates=gates,pass_all=all(gates.values())))
        for i,feature in enumerate(schema["ordered_features"]):
            xx=F[ids-1,i]
            if len(ids)>3:
                relations.append(dict(year=year,feature=feature["name"],
                    r_h1=float(pearsonr(xx[:h],yy[:h]).statistic),r_h2=float(pearsonr(xx[h:],yy[h:]).statistic),
                    rho_h1=float(spearmanr(xx[:h],yy[:h]).statistic),rho_h2=float(spearmanr(xx[h:],yy[h:]).statistic)))
    chosen=next((r for r in candidates if r["pass_all"]),None)
    split_ids={}
    if chosen:
        split_ids={"train":eligible(f'{chosen["year"]}-01-01T00:00:00+00:00',protocol["train_before"]),
                   "val":eligible(protocol["train_before"],protocol["validation_before"]),
                   "holdout":eligible(protocol["validation_before"],protocol["holdout_through"],True)}
    feasible=chosen is not None and len(split_ids["val"])>=100 and len(split_ids["holdout"])>=100
    decision={"protocol_hash":protocol_hash,"candidates":candidates,"chosen_year":None if chosen is None else chosen["year"],
        "training_allowed":feasible,"split_counts":{s:len(ids) for s,ids in split_ids.items()},
        "selection_basis":"Fixed training-only feasibility gates; validation/holdout structural counts only",
        "limitations":"No proof of stationarity or completeness; heldout data previously in original training catalog.",
        "reason":"PASS fixed data sufficiency checks" if feasible else "BLOCKED insufficient training stability/representation or purged evaluation sample count"}
    savej(OUT/"feasibility.json",decision)
    pd.DataFrame(relations).to_csv(OUT/"training_feature_stability.csv",index=False)
    fig,ax=plt.subplots(figsize=(8,4))
    for name,start,end in ranges:
        mask=cat.time<pd.Timestamp(end,tz="UTC")
        if start: mask &= cat.time>=pd.Timestamp(start,tz="UTC")
        vals=mag[mask]
        ax.hist(vals,bins=np.arange(1,9.01,.2),histtype="step",label=f"{name} n={len(vals)}")
    ax.set_yscale("log");ax.set_xlabel("Reported ML (training periods only)");ax.set_ylabel("Events per 0.2 ML bin");ax.legend(fontsize=7)
    fig.tight_layout();fig.savefig(OUT/"training_magnitude_frequency.png",dpi=140);plt.close(fig)
    print("Training-only feasibility:",json.dumps(clean(decision)))
else:
    decision=readj(OUT/"feasibility.json")
    print("Using cached analysis:",decision["reason"])


Using cached analysis: PASS fixed data sufficiency checks


In [3]:
if not cached:
    if decision["training_allowed"]:
        arrays={}; manifest_rows=[]; dependencies={}
        fit_ids=np.unique(np.concatenate([np.arange(i-20,i) for i in split_ids["train"]]))
        imputer=SimpleImputer(strategy="median").fit(F[fit_ids])
        scaler=StandardScaler().fit(imputer.transform(F[fit_ids]))
        transformed=scaler.transform(imputer.transform(F)).astype(np.float32)
        for split,ids in split_ids.items():
            arrays["X_"+split]=np.stack([transformed[i-20:i] for i in ids])
            arrays["y_"+split]=mag[ids].astype(np.float32)
            arrays["mean5_"+split]=np.array([mag[i-5:i].mean() for i in ids],dtype=np.float64)
            arrays["mean20_"+split]=np.array([mag[i-20:i].mean() for i in ids],dtype=np.float64)
            dependencies[split]=set()
            for position,i in enumerate(ids):
                start=int(min(i-39,left[i-20]))
                dependencies[split].update(range(start,i+1))
                manifest_rows.append(dict(split=split,sample_index=position,target_event_id=cat.source_id.iloc[i],
                    target_event_index=int(i),target_timestamp_utc=cat.time.iloc[i].isoformat(),
                    target_timestamp_original=cat.original_timestamp.iloc[i],target_magnitude=float(mag[i]),
                    input_event_ids=json.dumps(cat.source_id.iloc[i-20:i].tolist()),
                    input_timestamps_original=json.dumps(cat.original_timestamp.iloc[i-20:i].tolist()),
                    dependency_first_event_index=start,dependency_last_event_index=int(i),
                    dependency_first_timestamp=cat.time.iloc[start].isoformat(),
                    input_last_timestamp=cat.time.iloc[i-1].isoformat(),
                    target_gap_seconds=(cat.time.iloc[i]-cat.time.iloc[i-1]).total_seconds()))
        for a,b in [("train","val"),("train","holdout"),("val","holdout")]:
            assert not dependencies[a]&dependencies[b],f"Shared dependency events {a}/{b}"
        manifest_new=pd.DataFrame(manifest_rows)
        assert manifest_new.target_event_id.is_unique
        assert all(pd.Timestamp(r.input_last_timestamp)<pd.Timestamp(r.target_timestamp_utc) for r in manifest_new.itertuples())
        assert max(fit_ids)<min(split_ids["val"])
        for split in split_ids:
            assert arrays["X_"+split].shape==(len(split_ids[split]),20,10)
            assert np.isfinite(arrays["X_"+split]).all() and np.isfinite(arrays["y_"+split]).all()
            assert np.array_equal(arrays["y_"+split],mag[split_ids[split]].astype(np.float32))
            raw_last_mean=arrays["X_"+split][:,-1,6]*scaler.scale_[6]+scaler.mean_[6]
            assert np.allclose(raw_last_mean,arrays["mean20_"+split],atol=1e-6,rtol=0)
        np.savez_compressed(NEW/"modern_catalog_seq20.npz",**arrays)
        manifest_new.to_csv(NEW/"split_manifest.csv",index=False)
        pd.DataFrame({"event_index":fit_ids,"source_id":cat.source_id.iloc[fit_ids].to_numpy(),
                      "timestamp_utc":cat.time.iloc[fit_ids].astype(str).to_numpy()}).to_csv(NEW/"scaler_fit_events.csv",index=False)
        savej(NEW/"scalers.json",{"mean":scaler.mean_,"scale":scaler.scale_,"var":scaler.var_,
              "imputer_statistics":imputer.statistics_,"fit_event_count":len(fit_ids),"fit_scope":"Unique input events in new training partition only",
              "target_transform":{"name":"identity","scale":1,"offset":0}})
        savej(NEW/"feature_schema.json",schema)
        savej(NEW/"configuration.json",{"protocol":protocol,"protocol_hash":protocol_hash,"chosen_cutoff_year":decision["chosen_year"],
             "source_sha256":sha(SOURCE),"source_eligibility":"Original CSV verified against retained ML provenance; unchanged exclusion rules",
             "boundary_handling":"Drop targets until all20 inputs and rolling-feature dependency events lie in partition",
             "holdout_history":"Previously used by original model during training; retrospective reuse disclosed"})
        with np.load(NEW/"modern_catalog_seq20.npz",allow_pickle=False) as check:
            assert all(np.array_equal(check[k],v) for k,v in arrays.items())
        validation={"status":"PASS","split_counts":decision["split_counts"],"train_high":int((arrays["y_train"]>=4.5).sum()),
             "input_and_target_and_dependency_disjoint":True,"train_only_preprocessing":True,
             "all_inputs_before_targets":True,"finite_arrays":True,"target_identity_preserved":True,
             "roundtrip_npz_exact":True,"holdout_label_summaries":"Deferred until model selection is locked",
             "dataset_sha256":sha(NEW/"modern_catalog_seq20.npz"),"scaler_sha256":sha(NEW/"scalers.json")}
        savej(NEW/"validation.json",validation)
    assert all(sha(ROOT/rel)==digest for rel,digest in protected.items())
    savej(OUT/"analysis_complete.json",{"protocol_hash":protocol_hash,"training_allowed":decision["training_allowed"],
        "source_preservation":"PASS","source_sha256":sha(SOURCE),
        "artifacts":{p.name:sha(p) for p in NEW.iterdir() if p.is_file()}})
else:
    check=readj(OUT/"analysis_complete.json")
    assert all(sha(NEW/name)==digest for name,digest in check["artifacts"].items())
print("Dataset preparation:",decision["reason"],"; counts",decision["split_counts"])


Dataset preparation: PASS fixed data sufficiency checks ; counts {'train': 692, 'val': 346, 'holdout': 366}
